# 07 — Sampling (opcional)

**Objetivo:** avaliar se o desequilíbrio de classes exige tratamento.

**Princípios:**
1. **Medir antes de agir:** só aplicar reamostragem se houver desequilíbrio relevante (ex.: < 20% de eventos)
2. **RUS nunca na validação/teste:** preservar a prevalência real fora do treino
3. **Pesos de classe** são alternativa sem descartar dados
4. Se usar RUS, as probabilidades podem exigir **recalibração** para refletir a prevalência real

In [0]:
# ── Configuração central (idem notebook 00) ──
CATALOG = "workspace"
SCHEMA  = "ml_bakehouse"

from pyspark.sql import functions as F

train_base = spark.table(f"{CATALOG}.{SCHEMA}.train_base")

taxa = train_base.agg(F.avg("target").alias("taxa_evento")).collect()[0]["taxa_evento"]
contagem = train_base.groupBy("target").agg(F.count("*").alias("linhas")).orderBy("target")
display(contagem)
print(f"Taxa de evento no treino: {taxa:.2%}")

if taxa >= 0.2:
    print("➡️ Classes balanceadas (>= 20% de eventos): RUS NÃO é necessário.")
else:
    print("➡️ Desbalanceamento relevante: considerar RUS ou pesos de classe.")

In [0]:
# Implementação de RUS (executada apenas se necessária)
# Mantém TODOS os eventos e reduz a classe majoritária por amostragem estratificada
n_ev = train_base.filter("target = 1").count()
n_nev = train_base.filter("target = 0").count()
fracao_nev = n_ev / n_nev   # igualar as classes

train_rus = train_base.sampleBy("target", fractions={0: fracao_nev, 1: 1.0}, seed=42)

display(train_rus.groupBy("target").agg(F.count("*").alias("linhas")))

# Gravação — apenas se você decidir usá-lo no 08
(train_rus.write.mode("overwrite").option("overwriteSchema", "true")
     .saveAsTable(f"{CATALOG}.{SCHEMA}.train_rus_base"))
print(f"Salvo (opcional): {CATALOG}.{SCHEMA}.train_rus_base")

In [0]:
# Alternativa: pesos de classe (sem descartar dados)
# No pyspark.ml, use weightCol; aqui calculamos os pesos 1/n por classe
pesos = train_base.groupBy("target").count().collect()
n_total = train_base.count()
mapa_pesos = {r["target"]: n_total / (2 * r["count"]) for r in pesos}   # peso balanceado
print("Pesos por classe:", mapa_pesos)

train_weighted = train_base.withColumn(
    "class_weight", F.when(F.col("target") == 1, F.lit(mapa_pesos[1])).otherwise(F.lit(mapa_pesos[0]))
)
display(train_weighted.groupBy("target", "class_weight").agg(F.count("*").alias("linhas")))

## Recomendação para este projeto

Com taxa de evento de ~40% (**classes balanceadas**), a recomendação é:

* ❌ **Não aplicar RUS** — descartaria dados sem necessidade
* ✅ Manter a base original (`train_base`) no `08_modelagem`
* ✅ Se quiser exercitar as duas condições, compare `train_base` vs. `train_rus_base` e observe o impacto em **discriminação (AUC/KS)** e **calibração** — RUS tende a distorcer as probabilidades

**Validação e teste permanecem SEMPRE com a distribuição original.**